# Análisis de Hipótesis — Taco Bueno Pricing (H1–H8)

Notebook de **exploración** sobre las tablas Gold (`workspace.default.taco_bueno_gold_*`) generado por el pipeline [05_Pipeline_Job](#notebook-1988745176962912). Aquí se juega con los resultados y se validan las hipótesis del equipo, separado del pipeline productivo (03 Silver / 04 Gold).

Cada celda evalúa una hipótesis usando las tablas Gold (H7 requiere granularidad de orden y se calcula desde la Silver). El veredicto es **evidencia descriptiva**: confirma o refuta la consistencia con los datos, NO causalidad — para afirmar causa se necesita un experimento A/B o grupo de control.

Se pueden añadir celdas libremente: ninguna modifica tablas (solo lectura), así que este notebook nunca rompe el pipeline.

## Resumen de hipótesis

| # | Hipótesis | Estrategia propuesta | Fuente |
|---|---|---|---|
| H1 | Drive Thru no necesita descuentos | Mantenerlo sin promoción | gold_channel |
| H2 | Steak Soft Taco canibalizó su precio (~$233K a nivel ítem) | Revisar su descuento | gold_product |
| H3 | Recortar descuentos no causó la caída de tráfico | Buscar otras causas | gold_monthly |
| H4 | Oportunidad happy hour 1–3pm (valle post-lunch) | Promociones en esa ventana | gold_hourly |
| H5 | App: 50.9% con descuento en solo 7,486 transacciones | Repensar enfoque digital | gold_channel |
| H6 | Promos de fin de semana emparejan tráfico | Family meals y bundles | gold_dayofweek |
| H7 | Descuentos en órdenes pequeñas no recuperan revenue | Limitarlos | Silver (orden) |
| H8 | Descuentos en sides/condimentos son regalos (refutada a nivel ítem: ~$80K) | Re-análisis con canasta incremental | gold_product |

In [0]:
# H1 — "Drive Thru does not need discounts" → Estrategia: mantenerlo promotion-free
# Fase 3: montos a nivel ORDEN (revenue_gross/avg_order_value) y tasas por ORDEN
# (pct_orders_with_discount). discount_amount_item = descuento real a nivel ítem.
h1 = spark.sql("""
SELECT channel, transactions, orders, revenue_gross, avg_order_value,
       pct_orders_with_discount, discount_amount_check, discount_amount_item, active_stores
FROM workspace.default.taco_bueno_gold_channel
ORDER BY transactions DESC
""").toPandas()
display(h1)

dt = h1.loc[h1['channel'] == 'Drive Thru'].iloc[0]
mean_rate = h1['pct_orders_with_discount'].mean()
print(f"\nH1 — Evidencia (nueva base Gold, montos a nivel check):")
print(f"• Drive Thru: {dt['transactions']:,} líneas ({dt['transactions']/h1['transactions'].sum()*100:.1f}% del total) / {dt['orders']:,} órdenes, revenue bruto ${dt['revenue_gross']:,.0f}, check promedio ${dt['avg_order_value']}.")
print(f"• Tasa de descuento (ÓRDENES): {dt['pct_orders_with_discount']}% (promedio de canales: {mean_rate:.1f}%); descuento a nivel ítem: ${dt['discount_amount_item']:,.0f}.")
veredicto = "CONSISTENTE" if dt['pct_orders_with_discount'] < mean_rate else "NO CONSISTENTE"
print(f"VEREDICTO: {veredicto} con H1 — Drive Thru ya opera por debajo de la media de descuentos con $0 de descuento a nivel ítem. Nota: DoorDash tiene 0% de descuento, pero es un canal de terceros sin control de pricing; Drive Thru es el canal propio con menor descuento. (Descriptivo, no causal)")

In [0]:
# H2 — "Steak Soft Taco canibalizó su propio precio" → Revisar su descuento
# Fase 3: nivel ÍTEM. gross_item_sales = menu_price×qty; descuento = item_discount.
# El descuento a nivel check NO se atribuye al producto (la versión anterior sumaba
# total_discount por línea: el "$429K" era el techo de descuentos de check completos).
h2 = spark.sql("""
SELECT menu_item, report_group, transactions, units_sold, gross_item_sales,
       item_discount_amount, net_item_revenue, item_discount_per_unit,
       pct_lines_item_discount, pct_lines_in_discounted_check
FROM workspace.default.taco_bueno_gold_product
WHERE menu_item = 'Steak Soft Taco'
""").toPandas()
top_disc = spark.sql("""
SELECT menu_item, transactions, gross_item_sales, item_discount_amount,
       pct_lines_item_discount, item_discount_per_unit
FROM workspace.default.taco_bueno_gold_product
WHERE transactions >= 10000
ORDER BY pct_lines_item_discount DESC LIMIT 5
""").toPandas()
display(h2)
display(top_disc)

st = h2.iloc[0]
print(f"\nH2 — Evidencia (nivel ítem):")
print(f"• Steak Soft Taco: {st['transactions']:,} líneas, {st['units_sold']:,} unidades, ventas brutas ${st['gross_item_sales']:,.0f}.")
print(f"• {st['pct_lines_item_discount']}% de sus líneas llevan item_discount (la mayor del menú de alto volumen; #2: {top_disc.iloc[1]['menu_item']} con {top_disc.iloc[1]['pct_lines_item_discount']}%).")
print(f"• Descuento a nivel ítem: ${st['item_discount_amount']:,.0f} = {st['item_discount_amount']/st['gross_item_sales']*100:.0f}% de sus ventas brutas (${st['item_discount_per_unit']:.2f}/unidad).")
print(f"• {st['pct_lines_in_discounted_check']}% de sus líneas están en checks con descuento a nivel check — NO atribuible al producto.")
print("VEREDICTO: CONSISTENTE — el producto está sobre-descuentado a nivel ítem; revisar precio/eligibilidad del item_discount. La canibalización estricta (si el descuento quita venta a otros tacos) requiere análisis de canastas.")

In [0]:
# H3 — "Cutting discounts did not cause the traffic drop" → Buscar otras causas
# Fase 3: tasa de descuento por ORDEN (pct_orders_with_discount) y tráfico normalizado
# por tienda y por DÍA (la métrica mensual por tienda NO es por día — corregido).
h3 = spark.sql("""
SELECT month, transactions, orders, active_stores, pct_orders_with_discount,
       ROUND(orders / active_stores, 1) AS orders_per_store_month,
       ROUND(orders / (active_stores * DAY(LAST_DAY(month))), 1) AS orders_per_store_day
FROM workspace.default.taco_bueno_gold_monthly
WHERE month >= '2025-02-01' AND month < '2026-09-01'  -- excluye meses parciales (ene-25, sep-26)
ORDER BY month
""").toPandas()
h3['mes'] = h3['month'].dt.strftime('%Y-%m')
display(h3[['mes', 'transactions', 'orders_per_store_day', 'pct_orders_with_discount']])

corr_trans = h3['transactions'].corr(h3['pct_orders_with_discount'])
corr_store = h3['orders_per_store_day'].corr(h3['pct_orders_with_discount'])
h3['anio'] = h3['month'].dt.year
avg_25 = h3.loc[h3['anio'] == 2025, 'transactions'].mean()
avg_26 = h3.loc[h3['anio'] == 2026, 'transactions'].mean()
print(f"\nH3 — Evidencia (feb-2025 a ago-2026, meses completos):")
print(f"• Órdenes por tienda por DÍA: {h3['orders_per_store_day'].min():.0f}–{h3['orders_per_store_day'].max():.0f} (por mes: {h3['orders_per_store_month'].min():,.0f}–{h3['orders_per_store_month'].max():,.0f}).")
print(f"• Correlación mensual: tasa de descuento (órdenes) vs transacciones = {corr_trans:.2f}; vs órdenes/tienda/día = {corr_store:.2f}.")
print(f"• Transacciones promedio: 2025 = {avg_25/1e6:.2f}M/mes vs 2026 = {avg_26/1e6:.2f}M/mes ({(avg_26-avg_25)/avg_25*100:+.1f}%).")
if corr_trans < -0.5:
    v = "Los meses con MÁS descuento tuvieron MENOS transacciones — la evidencia apoya que el recorte no causó la caída."
elif corr_trans > 0.5:
    v = "Los meses con más descuento tuvieron más transacciones — NO se puede descartar que el recorte afectara el tráfico; requiere análisis por cohortes y estacionalidad."
else:
    v = "Correlación débil — el recorte de descuentos no explica por sí solo el tráfico; buscar otras causas (estacionalidad, aperturas/cierres, competencia)."
print(f"VEREDICTO: {v} (Correlacion temporal, no causal: no se controlaron variables externas como estacionalidad, aperturas/cierres o cambios de menu. Recomendamos A/B test para confirmar.)")

In [0]:
# H4 — "There is a happy hour opportunity from 1 to 3 pm" → Promociones en el valle post-lunch
# Corrección de auditoría (Fase 2): el revenue SUBE de 3-6 PM (pico de cena a las 17h);
# el valle real es 1-3 PM. Fase 3: montos a nivel ORDEN (revenue_gross) — los absolutos
# ya NO están inflados ×~6; antes solo las participaciones % eran válidas.
h4 = spark.sql("""
SELECT hour_of_day, transactions, orders, revenue_gross, avg_order_value,
       pct_orders_with_discount,
       ROUND(revenue_gross / SUM(revenue_gross) OVER (), 4) AS revenue_share
FROM workspace.default.taco_bueno_gold_hourly
ORDER BY hour_of_day
""").toPandas()
display(h4)

tarde = h4[h4['hour_of_day'].isin([13, 14, 15])]
pico = h4.loc[h4['revenue_gross'].idxmax()]
valle = h4.loc[h4['revenue_gross'].idxmin()]
media_desc = h4['pct_orders_with_discount'].mean()
print(f"\nH4 — Evidencia (corregido: valle post-lunch 1–3 PM, montos a nivel check):")
print(f"• Ventana 1–3pm (13–15h): {tarde['revenue_share'].sum()*100:.1f}% del revenue bruto, check promedio ${tarde['avg_order_value'].mean():.2f}.")
print(f"• Hora pico: {int(pico['hour_of_day'])}h (${pico['revenue_gross']/1e6:.1f}M). Hora valle: {int(valle['hour_of_day'])}h (${valle['revenue_gross']/1e6:.1f}M).")
print(f"• Tasa de descuento (órdenes) 13–15h: {tarde['pct_orders_with_discount'].mean():.2f}% vs media diaria {media_desc:.2f}%.")
comparacion = "por ENCIMA de la media" if tarde['pct_orders_with_discount'].mean() > media_desc else "por debajo de la media"
print(f"VEREDICTO: EVALUAR — la franja 13–15h es el valle post-lunch ({tarde['revenue_share'].sum()*100:.0f}% del revenue) con descuentos {comparacion} ({tarde['pct_orders_with_discount'].mean():.2f}% vs {media_desc:.2f}%): un happy hour dirigido a esta ventana podría aplanar la curva. Validar con A/B.")

In [0]:
# H5 — "App discounts 50.9% with only 7,486 transactions" → Rethink the digital approach
# Fase 3: montos a nivel ORDEN. El "$178,722 de revenue" de Mobile era la convención
# por línea (inflada); el revenue bruto real del canal se cuenta una vez por check.
h5 = spark.sql("""
SELECT channel, transactions, orders, revenue_gross, avg_order_value,
       pct_orders_with_discount, discount_amount_check, discount_amount_item
FROM workspace.default.taco_bueno_gold_channel
WHERE channel IN ('Mobile', 'Online', 'Online: DoorDash',
                  'Online: UberEats', 'Online: GrubHub', 'Online: Zipline')
ORDER BY transactions DESC
""").toPandas()
display(h5)

mob = h5[h5['channel'] == 'Mobile']
total_trans = h5['transactions'].sum()
if len(mob):
    m = mob.iloc[0]
    dd = h5.loc[h5['channel'] == 'Online: DoorDash'].iloc[0]
    print(f"\nH5 — Evidencia (montos a nivel check):")
    print(f"• Mobile: {m['transactions']:,} líneas / {m['orders']:,} órdenes ({m['transactions']/total_trans*100:.2f}% del digital), revenue bruto ${m['revenue_gross']:,.0f}, check promedio ${m['avg_order_value']}.")
    print(f"• {m['pct_orders_with_discount']}% de sus órdenes llevan descuento (todo a nivel check: item_discount = ${m['discount_amount_item']:,.0f}) vs DoorDash: {dd['orders']:,} órdenes con {dd['pct_orders_with_discount']}% de descuento.")
    veredicto = "CONSISTENTE" if m['pct_orders_with_discount'] > 30 else "REVISAR"
    print(f"VEREDICTO: {veredicto} con H5 — el volumen de app es marginal; su tasa de descuento elevada sugiere reorientar el incentivo digital a captación de primeras órdenes con cupones únicos medibles, en vez de descuento recurrente.")

In [0]:
# H6 — "Weekend promos even out traffic" → Family meals and bundles
# Fase 3: órdenes con clave store+date+check y tasa de descuento por ORDEN.
# El tráfico (órdenes/tienda/día) ya era correcto; cambia la tasa de descuento.
h6 = spark.sql("""
SELECT day_of_week_num, day_name, is_weekend, store_days, transactions, orders,
       avg_orders_per_store_day, revenue_gross, pct_orders_with_discount
FROM workspace.default.taco_bueno_gold_dayofweek
ORDER BY day_of_week_num
""").toPandas()
display(h6)

wknd = h6[h6['is_weekend']]
lab = h6[~h6['is_weekend']]
debil = h6.loc[h6['avg_orders_per_store_day'].idxmin()]
print(f"\nH6 — Evidencia:")
print(f"• Fin de semana: {wknd['avg_orders_per_store_day'].mean():.1f} órdenes/tienda/día vs laboral {lab['avg_orders_per_store_day'].mean():.1f} ({(wknd['avg_orders_per_store_day'].mean()-lab['avg_orders_per_store_day'].mean())/lab['avg_orders_per_store_day'].mean()*100:+.1f}%).")
print(f"• Día más débil: {debil['day_name']} ({debil['avg_orders_per_store_day']} órdenes/tienda/día).")
print(f"• Tasa de descuento (órdenes) finde {wknd['pct_orders_with_discount'].mean():.2f}% vs laboral {lab['pct_orders_with_discount'].mean():.2f}%.")
print(f"VEREDICTO: EVALUAR — un bundle/family meal dirigido al día más débil ({debil['day_name']}) tiene base de datos para 'emparejar' tráfico. Medir con grupo de control antes de generalizar la promo.")

In [0]:
# H7 — "Discounts on small orders do not make up for lost revenue" → Limitarlos
# Grain de ORDEN (store_id + date + check_number — el check_number se reutiliza entre
# fechas, por eso la clave incluye date; la versión anterior agrupaba solo store+check).
# Montos del check (MAX) tomados UNA vez por orden.
h7 = spark.sql("""
WITH ordenes AS (
  SELECT store_id, date, check_number,
         MAX(total) AS order_total,
         MAX(total_discount) AS order_discount,
         MAX(CASE WHEN total_discount > 0 THEN 1 ELSE 0 END) AS has_discount,
         COUNT(*) AS lines
  FROM workspace.default.taco_bueno_silver
  WHERE transaction_type != 'Void'
  GROUP BY store_id, date, check_number
)
SELECT CASE WHEN order_total < 10 THEN '1. <$10'
            WHEN order_total < 20 THEN '2. $10–19.99'
            WHEN order_total < 30 THEN '3. $20–29.99'
            WHEN order_total < 50 THEN '4. $30–49.99'
            ELSE '5. $50+' END AS order_bucket,
       COUNT(*) AS orders,
       ROUND(AVG(order_total), 2) AS avg_order_value,
       ROUND(SUM(has_discount) * 100.0 / COUNT(*), 2) AS pct_orders_with_discount,
       ROUND(AVG(CASE WHEN has_discount = 1 THEN order_discount END), 2) AS avg_discount_when_given,
       ROUND(SUM(CASE WHEN has_discount = 1 THEN order_discount ELSE 0 END), 0) AS discount_given_total
FROM ordenes
GROUP BY 1 ORDER BY 1
""").toPandas()
display(h7)

tot_desc = h7['discount_given_total'].sum()
small = h7[h7['order_bucket'] == '1. <$10'].iloc[0]
print(f"\nH7 — Evidencia (clave de orden corregida: store+date+check):")
print(f"• Órdenes <$10: {small['orders']:,} ({small['orders']/h7['orders'].sum()*100:.1f}% de las órdenes), {small['pct_orders_with_discount']}% reciben descuento, check bruto promedio ${small['avg_order_value']}.")
print(f"• Descuento dado en órdenes <$10: ${small['discount_given_total']:,.0f} ({small['discount_given_total']/tot_desc*100:.1f}% del descuento total a nivel orden ${tot_desc:,.0f}).")
print("VEREDICTO: EVALUAR — si el bucket pequeño concentra un % de descuento desproporcionado a su revenue, definir un umbral mínimo de compra (p. ej. $10) y validarlo con A/B antes de limitar.")

In [0]:
# H8 — "Discounts on sides and condiments are pure giveaways" → Stop discounting them
# Fase 3: descuentos a NIVEL ÍTEM (item_discount_amount). La versión anterior sumaba
# `total_discount` (monto del CHECK repetido por línea) y atribuía a estos grupos los
# descuentos de otros ítems del check ($5.26M atribuidos vs ~$80K reales a nivel ítem).
h8 = spark.sql("""
SELECT report_group,
       COUNT(*) AS products,
       SUM(transactions) AS transactions,
       ROUND(SUM(gross_item_sales), 0) AS gross_item_sales,
       ROUND(SUM(item_discount_amount), 0) AS item_discount_total,
       ROUND(SUM(item_discount_amount) * 100.0 / NULLIF(SUM(gross_item_sales), 0), 2) AS item_disc_pct_of_gross
FROM workspace.default.taco_bueno_gold_product
WHERE report_group LIKE '%Condiments%' OR report_group LIKE '%Sides%'
   OR report_group LIKE '%(I)%' OR report_group LIKE '%Substitution%'
GROUP BY report_group
ORDER BY item_discount_total DESC
""").toPandas()
display(h8)

tot8 = h8['item_discount_total'].sum()
print(f"\nH8 — Evidencia (descuentos a nivel ítem, nueva base Gold):")
for _, r in h8.iterrows():
    print(f"• {r['report_group']}: {r['transactions']:,} líneas, ${r['gross_item_sales']:,.0f} ventas brutas de ítem, ${r['item_discount_total']:,.0f} de item_discount ({r['item_disc_pct_of_gross']}% de sus ventas brutas).")
print(f"• Total descontado a nivel ÍTEM en sides/condimentos/ítems no-comida: ${tot8:,.0f}.")
print("VEREDICTO: REVISAR — la premisa 'giveaways' NO se sostiene a nivel ítem (el ~98% del monto de la versión anterior era descuento de check atribuido erróneamente por línea). La pregunta correcta es si los combos a nivel check dependen de estos acompañamientos; medir canasta incremental antes de cambiar la política.")